# Indian Fresher Job Market: Who Gets Hired & Why?

Every year, lakhs of fresh graduates apply across dozens of job platforms — LinkedIn, Naukri, Internshala, Unstop, campus placements, and more — with little visibility into what actually works. Which platform converts best? What profile gets hired? Where does the funnel break down?

this project takes a dataset of 5,000 fresher job applications spanning 20+ platforms and runs a full analytics lifecycle on it — from raw descriptive stats to  funnel diagnostics using Python (pandas, seaborn, , numpy), MySQL, and Power BI.

In [ ]:
from sqlalchemy import create_engine , text
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)

Total Tables: 1
Table Names:
- fresher_hiring_india_dataset


In [3]:
for table in tables:
    #    print(f"\n Table: {table}")
       query = text(f"SELECT COUNT(*) FROM {table}")
       df = pd.read_sql_query(query, engine)
       print(f"{table}", df.iloc[0,0])
       display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine))

fresher_hiring_india_dataset 5000


,candidate_id,full_name,gender,age,graduation_year,college,degree,branch,cgpa,backlogs,...,job_location,application_date,hiring_stage,interview_rounds,response_time_days,offered_salary_inr,linkedin_connections,profile_completion_pct,linkedin_premium,referral_applied
0,1,Deepika Shukla,Female,23,2021,BITS Goa,B.E.,Electrical,5.89,3,...,Bengaluru,2021-11-23,Technical Interview,3.0,22,None,1188,64,,No
1,2,Piyush Reddy,Male,25,2022,IIEST Shibpur,B.Sc (IT),Information Technology,8.78,0,...,Ahmedabad,2021-07-26,Shortlisted,NaN,30,None,2653,78,Yes,No
2,3,Itisha Arora,Female,21,2024,BIT Mesra Ranchi,BCA,Computer Applications,6.24,3,...,Delhi NCR,2022-03-09,Rejected,4.0,37,None,2990,75,,No
3,4,Darshan Dey,Male,22,2022,BITS Goa,B.Sc (CS),Computer Science,8.86,3,...,Coimbatore,2023-08-17,Online Assessment,NaN,36,None,97,98,No,No
4,5,Uma Grewal,Female,25,2020,Pune University,MCA,Computer Applications,7.54,2,...,Bengaluru,2024-05-11,Shortlisted,NaN,2,None,508,78,No,No


In [4]:
df = pd.read_sql_query(text("select * from fresher_hiring_india_dataset") , engine)

In [5]:
print(f"{df.info()}")
print("-"* 40)
print(f"null check : \n{df.isnull().sum()}")
print("-"* 40)
print(f"duplicate data : {df.duplicated().sum()}")
print("-"* 40)
print(f"data shape : \n{df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   candidate_id            5000 non-null   int64  
 1   full_name               5000 non-null   object 
 2   gender                  5000 non-null   object 
 3   age                     5000 non-null   int64  
 4   graduation_year         5000 non-null   int64  
 5   college                 5000 non-null   object 
 6   degree                  5000 non-null   object 
 7   branch                  5000 non-null   object 
 8   cgpa                    5000 non-null   float64
 9   backlogs                5000 non-null   int64  
 10  gap_year                5000 non-null   object 
 11  prior_internship        5000 non-null   object 
 12  projects_count          5000 non-null   int64  
 13  certifications_count    5000 non-null   int64  
 14  top_skills              5000 non-null   

In [6]:
df = df.rename(columns={
    "hiring_stage" : "current_hiring_stage",
    "interview_rounds" : "interview_rounds_completed"
    }
)

In [7]:
df['interview_rounds_completed'].min()

np.float64(1.0)

* fill with 0 as it is not possible to have negative interview rounds and also if the data is missing it means that the candidate has not completed any interview rounds yet.👇

* offered salary only have who offred or placed so we replace null with 0

In [8]:
df['interview_rounds_completed'] = df['interview_rounds_completed'].fillna(0)
df['offered_salary_inr'] = df['offered_salary_inr'].fillna(0)

In [9]:
print(f"null check after filling : \n{df.isnull().sum()}")

null check after filling : 
candidate_id                  0
full_name                     0
gender                        0
age                           0
graduation_year               0
college                       0
degree                        0
branch                        0
cgpa                          0
backlogs                      0
gap_year                      0
prior_internship              0
projects_count                0
certifications_count          0
top_skills                    0
platform                      0
company_applied               0
sector                        0
job_role                      0
work_type                     0
job_location                  0
application_date              0
current_hiring_stage          0
interview_rounds_completed    0
response_time_days            0
offered_salary_inr            0
linkedin_connections          0
profile_completion_pct        0
linkedin_premium              0
referral_applied              0
dtype: int64

In [10]:
# remove whitespace from object columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

In [11]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    if(len(vals) <=30):
        print(f"\n{col} ({len(vals)} unique): {sorted(vals)}")


gender (3 unique): ['Female', 'Male', 'Prefer not to say']

degree (10 unique): ['B.Com', 'B.E.', 'B.Sc (CS)', 'B.Sc (IT)', 'B.Tech', 'BBA', 'BCA', 'M.Tech', 'MBA', 'MCA']

branch (25 unique): ['AI/ML', 'Accounting', 'Business Administration', 'Business Analytics', 'Chemical Engineering', 'Civil', 'Civil Engineering', 'Commerce', 'Computer Applications', 'Computer Science', 'Data Science', 'Electrical', 'Electrical Engineering', 'Electronics', 'Electronics & Communication', 'Finance', 'HR', 'Information Technology', 'Marketing', 'Mechanical', 'Mechanical Engineering', 'Operations', 'Software Engineering', 'Strategy', 'VLSI']

gap_year (2 unique): ['No', 'Yes']

prior_internship (2 unique): ['No', 'Yes']

platform (20 unique): ['AngelList', 'Apna', 'Campus Placement', 'Cutshort', 'Foundit', 'Freshersworld', 'Glassdoor', 'HackerEarth', 'Hirist', 'IIMJobs', 'Indeed', 'Internshala', 'LinkedIn', 'Monster', 'Naukri', 'Referral', 'Shine', 'TimesJobs', 'Unstop', 'WorkIndia']

sector (14 uniqu

In [12]:
df['linkedin_premium'].value_counts()

linkedin_premium
       3901
No      867
Yes     232
Name: count, dtype: int64

* convert into "unknown" for linkedin premium column as 78 % of data is empty and we cannot drop or randomly fill with "No" as it will create bias in data

In [13]:
df['linkedin_premium'] = (
    df['linkedin_premium']
    .replace('', 'Unknown')
)

In [14]:
df['linkedin_premium'].value_counts()

linkedin_premium
Unknown    3901
No          867
Yes         232
Name: count, dtype: int64

In [15]:
df['application_date'] = pd.to_datetime(df['application_date'] , errors='coerce')

df['application_year'] = df['application_date'].dt.year
df['application_month'] = df['application_date'].dt.month
df['application_day'] = df['application_date'].dt.day

In [16]:
df.columns

Index(['candidate_id', 'full_name', 'gender', 'age', 'graduation_year',
       'college', 'degree', 'branch', 'cgpa', 'backlogs', 'gap_year',
       'prior_internship', 'projects_count', 'certifications_count',
       'top_skills', 'platform', 'company_applied', 'sector', 'job_role',
       'work_type', 'job_location', 'application_date', 'current_hiring_stage',
       'interview_rounds_completed', 'response_time_days',
       'offered_salary_inr', 'linkedin_connections', 'profile_completion_pct',
       'linkedin_premium', 'referral_applied', 'application_year',
       'application_month', 'application_day'],
      dtype='object')

In [16]:
df.to_sql(name="clean_dataset",
          con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print("clean dataset save in mysql")

clean dataset save in mysql


In [17]:
df_verify = pd.read_sql("SELECT * FROM clean_dataset LIMIT 5", engine)
display(df_verify)

,candidate_id,full_name,gender,age,graduation_year,college,degree,branch,cgpa,backlogs,...,interview_rounds_completed,response_time_days,offered_salary_inr,linkedin_connections,profile_completion_pct,linkedin_premium,referral_applied,application_year,application_month,application_day
0,1,Deepika Shukla,Female,23,2021,BITS Goa,B.E.,Electrical,5.89,3,...,3.0,22,0.0,1188,64,Unknown,No,2021,11,23
1,2,Piyush Reddy,Male,25,2022,IIEST Shibpur,B.Sc (IT),Information Technology,8.78,0,...,0.0,30,0.0,2653,78,Yes,No,2021,7,26
2,3,Itisha Arora,Female,21,2024,BIT Mesra Ranchi,BCA,Computer Applications,6.24,3,...,4.0,37,0.0,2990,75,Unknown,No,2022,3,9
3,4,Darshan Dey,Male,22,2022,BITS Goa,B.Sc (CS),Computer Science,8.86,3,...,0.0,36,0.0,97,98,No,No,2023,8,17
4,5,Uma Grewal,Female,25,2020,Pune University,MCA,Computer Applications,7.54,2,...,0.0,2,0.0,508,78,No,No,2024,5,11


In [18]:
original_count = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_dataset", engine)
print(f"✓ Rows in MySQL  : {original_count['cnt'][0]}")
print(f"✓ Rows in df     : {len(df)}")
print(f"✓ Match          : {original_count['cnt'][0] == len(df)}")

✓ Rows in MySQL  : 5000
✓ Rows in df     : 5000
✓ Match          : True


In [20]:
df.to_csv("clean_dataset.csv" , index=False)